<div style="background: linear-gradient(135deg, #0f1117 0%, #1a1d2e 100%); border-radius: 16px; padding: 48px 40px; margin-bottom: 8px; border: 1px solid #2e3350; position: relative; overflow: hidden;">
  <div style="position: absolute; top: -60px; right: -60px; width: 240px; height: 240px; background: radial-gradient(circle, rgba(79,142,247,0.08) 0%, transparent 70%); border-radius: 50%;"></div>
  <div style="display: inline-block; background: rgba(79,142,247,0.12); border: 1px solid rgba(79,142,247,0.3); color: #4f8ef7; font-size: 11px; font-weight: 700; letter-spacing: 2px; text-transform: uppercase; padding: 4px 14px; border-radius: 20px; margin-bottom: 20px; font-family: monospace;">Notebook 02 · Modelado</div>
  <h1 style="font-family: Georgia, serif; font-size: 36px; font-weight: 700; color: #e2e8f0; margin: 0 0 10px 0; letter-spacing: -0.5px;">Stroke <span style='color: #4f8ef7;'>Predictor</span></h1>
  <p style="color: #8892a4; font-size: 15px; margin: 0 0 28px 0; font-family: monospace;">Pipeline · Modelos · MLflow Tracking</p>
  <div style="display: flex; gap: 32px; flex-wrap: wrap;">
    <div><div style="font-size: 10px; color: #8892a4; text-transform: uppercase; letter-spacing: 1px;">Autor</div><div style="font-size: 14px; color: #e2e8f0; font-weight: 600;">[Tu nombre]</div></div>
    <div><div style="font-size: 10px; color: #8892a4; text-transform: uppercase; letter-spacing: 1px;">Fecha</div><div style="font-size: 14px; color: #e2e8f0; font-weight: 600;">Abril 2026</div></div>
    <div><div style="font-size: 10px; color: #8892a4; text-transform: uppercase; letter-spacing: 1px;">Dataset</div><div style="font-size: 14px; color: #e2e8f0; font-weight: 600;">stroke_dataset.csv</div></div>
    <div><div style="font-size: 10px; color: #8892a4; text-transform: uppercase; letter-spacing: 1px;">Experimento MLflow</div><div style="font-size: 14px; color: #4f8ef7; font-weight: 600;">stroke_project</div></div>
  </div>
</div>

<div style="background: rgba(239,68,68,0.06); border-left: 3px solid #ef4444; border-radius: 0 8px 8px 0; padding: 14px 18px; margin: 8px 0;">
  <strong style="color: #ef4444; font-size: 12px;">⚠️ CONTEXTO CRÍTICO</strong><br>
  <span style="color: #94a3b8; font-size: 13px;">El dataset tiene un desbalance de <strong style='color:#e2e8f0;'>19:1</strong> (95% sin ictus / 5% con ictus). Accuracy no es una métrica válida. Usamos <strong style='color:#e2e8f0;'>AUC-ROC como métrica principal</strong> y <strong style='color:#e2e8f0;'>Recall como métrica secundaria</strong> para minimizar falsos negativos.</span>
</div>

---

## Índice
1. Setup e importaciones  
2. Configuración de MLflow  
3. Carga y versiones del dataset  
4. Preprocesado con ColumnTransformer  
5. Pipeline de entrenamiento  
6. Función principal `run_experiment`  
7. Ejecución de experimentos  
8. Comparativa de resultados

<div style="background:#1a1d2e; border-radius:10px; padding:20px 24px; border:1px solid #2e3350; margin-bottom:4px;">
  <div style="font-size:11px; color:#4f8ef7; text-transform:uppercase; letter-spacing:1.5px; font-weight:700; margin-bottom:8px;">01 · Setup e importaciones</div>
  <p style="color:#8892a4; font-size:13px; margin:0;">Cargamos todas las dependencias necesarias. Hay dos puntos clave aquí:<br><br>
  • Usamos <code>imblearn.pipeline.Pipeline</code> en lugar de <code>sklearn.pipeline.Pipeline</code> — esto es fundamental porque la versión de imbalanced-learn sabe que SMOTE solo debe aplicarse durante el <code>fit()</code>, nunca durante el <code>predict()</code> ni en el test set.<br><br>
  • <code>warnings.filterwarnings('ignore')</code> limpia la salida del notebook — en producción esto nunca se haría.</p>
</div>

In [ ]:
print("✓ Iniciando kernel...")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ML — sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    recall_score,
    f1_score,
    precision_score,
    confusion_matrix,
    RocCurveDisplay
)

# imblearn — Pipeline que entiende SMOTE
# ⚠️ IMPORTANTE: usar imblearn.pipeline, NO sklearn.pipeline
# La versión de imblearn garantiza que SMOTE solo se aplica en train, nunca en test
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

# MLflow
import mlflow
import mlflow.sklearn

import warnings
warnings.filterwarnings("ignore")

# ── Constantes globales ──────────────────────────────────────────
RANDOM_STATE = 42
TEST_SIZE    = 0.2

print("✓ Importaciones completadas")

<div style="background:#1a1d2e; border-radius:10px; padding:20px 24px; border:1px solid #2e3350; margin-bottom:4px;">
  <div style="font-size:11px; color:#4f8ef7; text-transform:uppercase; letter-spacing:1.5px; font-weight:700; margin-bottom:8px;"></div>
  <p style="color:#8892a4; font-size:13px; margin:0;">
  MLflow es un sistema de tracking de experimentos. Cada vez que entrenas un modelo, MLflow guarda automáticamente los parámetros, métricas y el modelo serializado. Esto te permite:<br><br>
  • <strong style='color:#e2e8f0;'>Reproducir</strong> cualquier experimento anterior con exactamente los mismos parámetros<br>
  • <strong style='color:#e2e8f0;'>Comparar</strong> todos los modelos del equipo en una sola tabla<br>
  • <strong style='color:#e2e8f0;'>Versionar</strong> los modelos y descargar el mejor para producción<br><br>
  Para ver la UI ejecuta en terminal: <code>mlflow ui</code> y abre <code>http://localhost:5000</code>
  </p>
</div>

In [ ]:
# ── Configuración MLflow ─────────────────────────────────────────
# tracking_uri: dónde se guardan los experimentos (carpeta local mlruns/)
# Ajusta la ruta a tu entorno WSL
MLFLOW_URI = "file:///mnt/c/Users/under/Documents/F5/3_Projects/Stroker_project/mlruns"

# ⚠️ ACUERDO DE EQUIPO: todos deben usar exactamente este nombre
# Si uno usa 'stroke_project' y otro 'Stroke_Project', son experimentos distintos
EXPERIMENT_NAME = "stroke_project"

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"✓ MLflow configurado")
print(f"  Tracking URI : {MLFLOW_URI}")
print(f"  Experimento  : {EXPERIMENT_NAME}")
print(f"  UI           : mlflow ui → http://localhost:5001")

<div style="background:#1a1d2e; border-radius:10px; padding:20px 24px; border:1px solid #2e3350; margin-bottom:4px;">
  <div style="font-size:11px; color:#4f8ef7; text-transform:uppercase; letter-spacing:1.5px; font-weight:700; margin-bottom:8px;">03 · Carga y versiones del dataset</div>
  <p style="color:#8892a4; font-size:13px; margin:0;">
  <code>get_dataset(version)</code> centraliza toda la lógica de limpieza en un solo lugar. 
  Ventaja: si el equipo decide cambiar cómo se trata <code>work_type='children'</code>, 
  solo hay que modificar esta función — todos los experimentos se actualizan automáticamente.<br><br>
  Ofrecemos dos versiones del dataset:<br>
  • <strong style='color:#e2e8f0;'>full</strong>: todos los pacientes (incluye menores de 17 años)<br>
  • <strong style='color:#e2e8f0;'>adults</strong>: solo pacientes ≥ 17 años (decisión del EDA)
  </p>
</div>

In [ ]:
# Carga inicial — el raw nunca se modifica
df = pd.read_csv("../data/raw/stroke_dataset.csv")
print(f"✓ Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"  Stroke=1 : {df['stroke'].sum()} ({df['stroke'].mean()*100:.1f}%)")
print(f"  Stroke=0 : {(df['stroke']==0).sum()} ({(1-df['stroke'].mean())*100:.1f}%)")

In [ ]:
def get_dataset(version: str = "full") -> pd.DataFrame:
    """
    Devuelve el dataset con las limpiezas acordadas en el EDA.
    
    Parámetros
    ----------
    version : str
        'full'   → todos los pacientes
        'adults' → solo pacientes con age >= 17
    
    Retorna
    -------
    pd.DataFrame con las transformaciones aplicadas
    
    Decisiones del EDA aplicadas aquí
    ----------------------------------
    - work_type='children' → 'not_applied' (no tiene sentido laboral para menores)
    - smoking_status='Unknown' se mantiene como categoría implícita de faltante
    """
    assert version in ("full", "adults"), f"version debe ser 'full' o 'adults', recibido: {version}"
    
    df_copy = df.copy()
    
    # Decisión EDA: work_type='children' → 'not_applied'
    df_copy.loc[df_copy["work_type"] == "children", "work_type"] = "not_applied"
    
    # Decisión EDA: filtrar menores si se pide versión adultos
    if version == "adults":
        before = len(df_copy)
        df_copy = df_copy[df_copy["age"] >= 17].copy()
        print(f"  Filtro adultos: {before - len(df_copy)} filas eliminadas")
    
    print(f"✓ Dataset '{version}': {df_copy.shape[0]:,} filas")
    return df_copy

<div style="background:#1a1d2e; border-radius:10px; padding:20px 24px; border:1px solid #2e3350; margin-bottom:4px;">
  <div style="font-size:11px; color:#4f8ef7; text-transform:uppercase; letter-spacing:1.5px; font-weight:700; margin-bottom:8px;">04 · Preprocesado con ColumnTransformer</div>
  <p style="color:#8892a4; font-size:13px; margin:0;">
  <code>ColumnTransformer</code> aplica transformaciones distintas a cada grupo de columnas en paralelo:<br><br>
  • <strong style='color:#e2e8f0;'>Numéricas</strong> → <code>StandardScaler</code>: centra en media 0 y escala a desviación 1. Necesario para Logistic Regression (sensible a la escala). Random Forest y XGBoost no lo necesitan pero no les perjudica.<br>
  • <strong style='color:#e2e8f0;'>Categóricas</strong> → <code>OneHotEncoder</code>: convierte categorías en columnas binarias. <code>handle_unknown='ignore'</code> evita errores si en producción llega una categoría no vista en entrenamiento.<br><br>
  <strong style='color:#ef4444;'>⚠️ Bug corregido:</strong> <code>build_preprocessor</code> recibe <code>X_train</code>, no <code>X</code> completo. El preprocesador nunca debe ver el test set.
  </p>
</div>

In [ ]:
def build_preprocessor(X_train: pd.DataFrame) -> ColumnTransformer:
    """
    Construye el preprocesador sobre X_train únicamente.
    
    ⚠️ Recibe X_train, no X completo.
    El fit del ColumnTransformer aprenderá media, std y categorías
    solo del conjunto de entrenamiento — nunca del test.
    
    Parámetros
    ----------
    X_train : pd.DataFrame
        Conjunto de entrenamiento (sin la columna target)
    
    Retorna
    -------
    ColumnTransformer sin fitear (el Pipeline lo fiteará en .fit())
    """
    cat_cols = X_train.select_dtypes(include="object").columns.tolist()
    num_cols = X_train.select_dtypes(exclude="object").columns.tolist()
    
    print(f"  Columnas numéricas  : {num_cols}")
    print(f"  Columnas categóricas: {cat_cols}")
    
    preprocessor = ColumnTransformer([
        ("num", StandardScaler(),                        num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"),  cat_cols),
    ])
    
    return preprocessor

<div style="background:#1a1d2e; border-radius:10px; padding:20px 24px; border:1px solid #2e3350; margin-bottom:4px;">
  <div style="font-size:11px; color:#4f8ef7; text-transform:uppercase; letter-spacing:1.5px; font-weight:700; margin-bottom:8px;">05 · Pipeline de entrenamiento</div>
  <p style="color:#8892a4; font-size:13px; margin:0;">
  El Pipeline encadena pasos en orden. Cuando llamas a <code>pipeline.fit(X_train, y_train)</code>, cada paso se ejecuta en secuencia:<br><br>
  <code style='background:#161925; padding:2px 6px; border-radius:4px;'>prep → [smote] → model</code><br><br>
  La magia del Pipeline de imblearn es que durante el <code>predict()</code>, SMOTE simplemente no se ejecuta — solo actúa en el <code>fit()</code>. Esto garantiza que el test set nunca es modificado.
  </p>
</div>

In [ ]:
def build_pipeline(model, preprocessor, use_smote: bool = False) -> Pipeline:
    """
    Construye el Pipeline de entrenamiento.
    
    Orden de pasos:
        1. prep  : ColumnTransformer (escala + encoding)
        2. smote : SMOTE (solo si use_smote=True, solo en fit)
        3. model : clasificador
    
    ⚠️ Usamos imblearn.pipeline.Pipeline, no sklearn.pipeline.Pipeline.
    La versión de imblearn es compatible con SMOTE y garantiza
    que el resampleo solo ocurre durante el entrenamiento.
    
    Parámetros
    ----------
    model        : estimador sklearn compatible
    preprocessor : ColumnTransformer ya construido
    use_smote    : bool — si True, añade SMOTE al pipeline
    """
    steps = [("prep", preprocessor)]
    
    if use_smote:
        steps.append(("smote", SMOTE(random_state=RANDOM_STATE)))
    
    steps.append(("model", model))
    
    return Pipeline(steps)

<div style="background:#1a1d2e; border-radius:10px; padding:20px 24px; border:1px solid #2e3350; margin-bottom:4px;">
  <div style="font-size:11px; color:#4f8ef7; text-transform:uppercase; letter-spacing:1.5px; font-weight:700; margin-bottom:8px;">06 · Función principal run_experiment</div>
  <p style="color:#8892a4; font-size:13px; margin:0;">
  Esta función orquesta todo el flujo de un experimento:<br><br>
  <strong style='color:#e2e8f0;'>Métricas que logueamos en MLflow:</strong><br>
  • <code>auc</code> — métrica principal, robusta al desbalance<br>
  • <code>recall</code> — minimizar falsos negativos (crítico en contexto médico)<br>
  • <code>f1</code> — balance entre precision y recall<br>
  • <code>precision</code> — de los que predecimos positivo, cuántos lo son<br><br>
  <strong style='color:#e2e8f0;'>Tags vs Params en MLflow:</strong><br>
  • <code>mlflow.log_param()</code> → hiperparámetros del modelo (numéricos, comparables)<br>
  • <code>mlflow.set_tag()</code> → metadatos del experimento (autor, dataset, versión)
  </p>
</div>

In [ ]:
def run_experiment(
    model,
    model_name: str,
    dataset_version: str = "full",
    use_smote: bool = False,
    author: str = "[tu nombre]"
) -> dict:
    """
    Entrena un modelo, evalúa y registra todo en MLflow.
    
    Parámetros
    ----------
    model           : estimador sklearn (LogisticRegression, RandomForest, etc.)
    model_name      : str — nombre del run en MLflow
    dataset_version : str — 'full' o 'adults'
    use_smote       : bool — si True aplica SMOTE en entrenamiento
    author          : str — nombre del miembro del equipo
    
    Retorna
    -------
    dict con métricas del experimento (para comparativa al final)
    """
    print(f"\n{'='*60}")
    print(f"  {model_name} | dataset={dataset_version} | SMOTE={use_smote}")
    print(f"{'='*60}")
    
    # ── 1. Datos ──────────────────────────────────────────────────
    df_exp = get_dataset(dataset_version)
    X = df_exp.drop("stroke", axis=1)
    y = df_exp["stroke"]
    
    # Split estratificado: garantiza misma proporción de clases en train y test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        stratify=y,          # ← fundamental con datos desbalanceados
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE
    )
    
    print(f"  Train: {len(X_train):,} filas | Test: {len(X_test):,} filas")
    print(f"  Stroke en train: {y_train.sum()} ({y_train.mean()*100:.1f}%)")
    
    # ── 2. Pipeline ───────────────────────────────────────────────
    # ⚠️ build_preprocessor recibe X_train, no X
    preprocessor = build_preprocessor(X_train)
    pipeline = build_pipeline(model, preprocessor, use_smote)
    
    # ── 3. Entrenamiento + evaluación dentro del run de MLflow ────
    with mlflow.start_run(run_name=f"{model_name}_{dataset_version}_smote={use_smote}"):
        
        # Entrenamiento
        pipeline.fit(X_train, y_train)
        
        # Predicciones
        y_pred = pipeline.predict(X_test)
        y_prob = pipeline.predict_proba(X_test)[:, 1]
        
        # ── Métricas ──
        auc       = roc_auc_score(y_test, y_prob)
        recall    = recall_score(y_test, y_pred)
        f1        = f1_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        
        # ── Log métricas (acuerdo de equipo: estos nombres exactos) ──
        mlflow.log_metric("auc",       round(auc, 4))
        mlflow.log_metric("recall",    round(recall, 4))
        mlflow.log_metric("f1",        round(f1, 4))
        mlflow.log_metric("precision", round(precision, 4))
        
        # ── Log hiperparámetros del modelo ──
        # get_params() devuelve todos los parámetros automáticamente
        mlflow.log_params(model.get_params())
        
        # ── Tags: metadatos del experimento (no son hiperparámetros) ──
        mlflow.set_tag("author",          author)
        mlflow.set_tag("dataset_version", dataset_version)
        mlflow.set_tag("use_smote",       str(use_smote))
        mlflow.set_tag("model_type",      model_name)
        
        # ── Guardar el pipeline completo (preprocesado + modelo) ──
        mlflow.sklearn.log_model(pipeline, artifact_path=model_name)
        
        # ── Output en notebook ──
        print(f"\n  AUC-ROC   : {auc:.4f}  ← métrica principal")
        print(f"  Recall    : {recall:.4f}  ← minimizar falsos negativos")
        print(f"  F1-score  : {f1:.4f}")
        print(f"  Precision : {precision:.4f}")
        print(f"\n{classification_report(y_test, y_pred, target_names=['Sin ictus', 'Ictus'])}")
        
        print(f"  ✓ Run registrado en MLflow")
        
        return {
            "modelo":   model_name,
            "dataset":  dataset_version,
            "smote":    use_smote,
            "auc":      round(auc, 4),
            "recall":   round(recall, 4),
            "f1":       round(f1, 4),
            "precision":round(precision, 4),
        }

<div style="background:#1a1d2e; border-radius:10px; padding:20px 24px; border:1px solid #2e3350; margin-bottom:4px;">
  <div style="font-size:11px; color:#4f8ef7; text-transform:uppercase; letter-spacing:1.5px; font-weight:700; margin-bottom:8px;">07 · Ejecución de experimentos</div>
  <p style="color:#8892a4; font-size:13px; margin:0;">
  Aquí definimos los modelos y lanzamos los experimentos. Cada llamada a <code>run_experiment</code> genera un run independiente en MLflow con sus propias métricas, parámetros y el modelo serializado.<br><br>
  Guardamos todos los resultados en una lista para generar la comparativa al final.
  </p>
</div>

In [ ]:
# ── Definición de modelos ────────────────────────────────────────
# class_weight='balanced' ajusta los pesos inversamente proporcional
# a la frecuencia de cada clase. Es la forma más simple de combatir
# el desbalance sin modificar el dataset.

lr_model = LogisticRegression(
    class_weight="balanced",  # compensa el desbalance 19:1
    max_iter=1000,             # más iteraciones para datasets con escalas distintas
    random_state=RANDOM_STATE
)

rf_model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=RANDOM_STATE
)

print("✓ Modelos definidos")
print(f"  Logistic Regression  : {lr_model.get_params()}")
print(f"  Random Forest        : {rf_model.get_params()}")

In [ ]:
# ── Lanzamiento de experimentos ──────────────────────────────────
# Guardamos cada resultado para la comparativa final
# Cambia 'author' por tu nombre
AUTHOR = "Jonathan"

results = []

# Baseline: Logistic Regression sin SMOTE
results.append(run_experiment(lr_model, "LogisticRegression", "full",   use_smote=False, author=AUTHOR))

# Logistic Regression con SMOTE
results.append(run_experiment(lr_model, "LogisticRegression", "full",   use_smote=True,  author=AUTHOR))

# Random Forest sin SMOTE
results.append(run_experiment(rf_model, "RandomForest",       "full",   use_smote=False, author=AUTHOR))

# Random Forest con SMOTE
results.append(run_experiment(rf_model, "RandomForest",       "full",   use_smote=True,  author=AUTHOR))

# Solo adultos — ¿mejora el modelo sin los menores?
results.append(run_experiment(lr_model, "LogisticRegression", "adults", use_smote=False, author=AUTHOR))
results.append(run_experiment(rf_model, "RandomForest",       "adults", use_smote=True,  author=AUTHOR))

<div style="background:#1a1d2e; border-radius:10px; padding:20px 24px; border:1px solid #2e3350; margin-bottom:4px;">
  <div style="font-size:11px; color:#4f8ef7; text-transform:uppercase; letter-spacing:1.5px; font-weight:700; margin-bottom:8px;">08 · Comparativa de resultados</div>
  <p style="color:#8892a4; font-size:13px; margin:0;">
  Comparamos todos los experimentos localmente. La comparativa completa con los modelos de tus compañeros la harás desde la UI de MLflow (<code>mlflow ui</code>), donde puedes filtrar, ordenar y visualizar curvas ROC de todos los runs en paralelo.
  </p>
</div>

In [ ]:
# ── Tabla comparativa local ───────────────────────────────────────
results_df = pd.DataFrame(results)

# Ordenar por AUC descendente (métrica principal)
results_df = results_df.sort_values("auc", ascending=False).reset_index(drop=True)

# Highlight del mejor por columna
print("\n📊 COMPARATIVA DE EXPERIMENTOS (ordenado por AUC-ROC)\n")
display(
    results_df.style
    .background_gradient(subset=["auc", "recall", "f1"], cmap="YlGn")
    .format({"auc": "{:.4f}", "recall": "{:.4f}", "f1": "{:.4f}", "precision": "{:.4f}"})
    .set_caption("Verde más oscuro = mejor valor")
)

# Mejor modelo según AUC
best = results_df.iloc[0]
print(f"\n✓ Mejor modelo: {best['modelo']} | dataset={best['dataset']} | SMOTE={best['smote']}")
print(f"  AUC-ROC : {best['auc']:.4f}")
print(f"  Recall  : {best['recall']:.4f}")

In [ ]:
# ── Visualización: AUC por experimento ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0f1117')

for ax in axes:
    ax.set_facecolor('#1a1d2e')
    ax.tick_params(colors='#8892a4')
    ax.spines[:].set_color('#2e3350')

# Etiquetas del eje X
labels = [f"{r['modelo'].replace('LogisticRegression','LR').replace('RandomForest','RF')}\n{r['dataset']} SMOTE={r['smote']}" 
          for _, r in results_df.iterrows()]

colors = ['#4f8ef7' if not r['smote'] else '#7c3aed' for _, r in results_df.iterrows()]

# AUC
axes[0].barh(labels, results_df['auc'], color=colors, alpha=0.85)
axes[0].axvline(0.85, color='#22c55e', linestyle='--', alpha=0.7, label='Objetivo ≥ 0.85')
axes[0].set_title('AUC-ROC por experimento', color='#e2e8f0', fontsize=13)
axes[0].set_xlabel('AUC-ROC', color='#8892a4')
axes[0].legend(labelcolor='#8892a4', facecolor='#1a1d2e', edgecolor='#2e3350')
axes[0].set_xlim(0, 1)

# Recall
axes[1].barh(labels, results_df['recall'], color=colors, alpha=0.85)
axes[1].axvline(0.75, color='#22c55e', linestyle='--', alpha=0.7, label='Objetivo ≥ 0.75')
axes[1].set_title('Recall (clase positiva) por experimento', color='#e2e8f0', fontsize=13)
axes[1].set_xlabel('Recall', color='#8892a4')
axes[1].legend(labelcolor='#8892a4', facecolor='#1a1d2e', edgecolor='#2e3350')
axes[1].set_xlim(0, 1)

plt.suptitle('Comparativa de experimentos — Stroke Predictor', 
             color='#e2e8f0', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../reports/comparativa_modelos.png', dpi=150, bbox_inches='tight',
            facecolor='#0f1117')
plt.show()
print("✓ Gráfico guardado en reports/comparativa_modelos.png")

<div style="background:#1a1d2e; border-radius:10px; padding:20px 24px; border:1px solid #2e3350; margin-top: 8px;">
  <div style="font-size:11px; color:#22c55e; text-transform:uppercase; letter-spacing:1.5px; font-weight:700; margin-bottom:12px;">✓ Siguientes pasos</div>
  <div style="display:flex; flex-direction:column; gap:10px;">
    <div style="display:flex; gap:12px; align-items:flex-start;">
      <div style="width:22px; height:22px; border-radius:50%; background:rgba(79,142,247,0.2); color:#4f8ef7; font-size:11px; font-weight:700; display:flex; align-items:center; justify-content:center; flex-shrink:0;">1</div>
      <div style="color:#e2e8f0; font-size:13px;">Ejecutar <code>mlflow ui</code> en terminal y comparar todos los runs del equipo en <code>http://localhost:5000</code></div>
    </div>
    <div style="display:flex; gap:12px; align-items:flex-start;">
      <div style="width:22px; height:22px; border-radius:50%; background:rgba(79,142,247,0.2); color:#4f8ef7; font-size:11px; font-weight:700; display:flex; align-items:center; justify-content:center; flex-shrink:0;">2</div>
      <div style="color:#e2e8f0; font-size:13px;">Añadir XGBoost + Optuna en este mismo notebook como tercer modelo</div>
    </div>
    <div style="display:flex; gap:12px; align-items:flex-start;">
      <div style="width:22px; height:22px; border-radius:50%; background:rgba(79,142,247,0.2); color:#4f8ef7; font-size:11px; font-weight:700; display:flex; align-items:center; justify-content:center; flex-shrink:0;">3</div>
      <div style="color:#e2e8f0; font-size:13px;">Seleccionar el mejor modelo y refactorizarlo a <code>src/models/train.py</code></div>
    </div>
  </div>
</div>